# BiciTierra Market - Segmentacio

Objectiu: preparar una lectura útil de perfils de client abans de passar a un clustering més formal.

## Dades

Hi ha 2.000 clients sintètics, una fotografia de 30/06/2026 independent de la taula de vendes. Per a un clustering, excloeu identificador i data, escaleu magnituds i decidiu com tractar els nuls. `annual_spend`, `avg_order_value` i `orders_last_12m` tenen una relació aritmètica: usar-les totes pot donar massa pes al valor monetari. La segmentació per regles és només una referència.

## Ruta de treball

1. Triar variables rellevants per al valor del client.
2. Detectar casos amb dades absents o delicades.
3. Construir una segmentació simple orientada a negoci.
4. Escriure què validaries després amb clustering.

In [ ]:
from pathlib import Path
import pandas as pd

CUSTOMERS = Path('..') / 'dades' / 'customers.csv'
customers = pd.read_csv(CUSTOMERS)
customers[['customer_id', 'annual_spend', 'orders_last_12m', 'returns_ratio', 'bike_interest']].head()

## 1. Revisió de qualitat

No val segmentar sense mirar abans si hi ha dades que exigixen tractament.

In [ ]:
customers.isna().sum().rename('nulls')

In [ ]:
customers[['annual_spend', 'avg_order_value', 'orders_last_12m', 'returns_ratio']].describe().round(2)

## 2. Segmentació simple orientada a negoci

Mentres no usem clustering formal, podem construir una primera lectura amb regles transparents.

In [ ]:
def simple_segment(row):
    if pd.isna(row['annual_spend']):
        return 'revisar_dada'
    if row['annual_spend'] >= 3000 and row['returns_ratio'] <= 0.02:
        return 'premium'
    if row['annual_spend'] >= 1200 and row['orders_last_12m'] >= 8:
        return 'valor_mitja_alt'
    return 'ocasional_o_baix_valor'

customers['simple_segment'] = customers.apply(simple_segment, axis=1)
customers[['customer_id', 'annual_spend', 'orders_last_12m', 'returns_ratio', 'simple_segment']].sort_values(['simple_segment', 'annual_spend'], ascending=[True, False])

In [ ]:
customers['simple_segment'].value_counts().rename_axis('segment').to_frame('clients')

## 3. Lectura per segment

Si la segmentació no permet una interpretació comercial, encara no està madura.

In [ ]:
customers.groupby('simple_segment')[['annual_spend', 'avg_order_value', 'orders_last_12m', 'returns_ratio']].mean().round(2)

## 4. Què faltaria validar amb clustering real

- si els grups continuen apareixent sense regles manuals
- si hi ha clients sorollosos o fronterers
- si la lectura comercial es manté quan passem a un algorisme de segmentació

In [ ]:
customers[['annual_spend','orders_last_12m','days_since_last_order','returns_ratio']].corr().round(2)